In [1]:
import pandas as pd
import numpy as np

# Load data

In [2]:
df = pd.read_csv('/Users/huynhphuongchi/Desktop/Unipi/DSS/Module 2/LDS Data 2025-2026/tracks_cleaned.csv') 

# General check

In [3]:
print("df_tracks:",df.shape)

df_tracks: (10515, 36)


In [4]:
10515/11154*100

94.27111350188272

In [5]:
df.isna().sum()

id                      0
id_artist               0
title                   0
featured_artists        0
primary_artist          0
language                0
album                   0
swear_IT                0
swear_EN                0
swear_IT_words          0
swear_EN_words          0
year                    0
month                   0
day                     0
n_sentences             0
n_tokens                0
char_per_tok            0
avg_token_per_clause    0
bpm                     0
rolloff                 0
flux                    0
flatness                0
spectral_complexity     0
pitch                   0
loudness                0
album_name              0
album_release_date      0
album_type              0
disc_number             0
track_number            0
duration_ms             0
explicit                0
popularity              0
id_album                0
lyrics                  0
streams@1month          0
dtype: int64

In [6]:
sorted(df["day"].unique())
sorted(df["month"].unique())
sorted(df["year"].unique())

[np.float64(1992.0),
 np.float64(1993.0),
 np.float64(1994.0),
 np.float64(1995.0),
 np.float64(1996.0),
 np.float64(1997.0),
 np.float64(1998.0),
 np.float64(1999.0),
 np.float64(2000.0),
 np.float64(2001.0),
 np.float64(2002.0),
 np.float64(2003.0),
 np.float64(2004.0),
 np.float64(2005.0),
 np.float64(2006.0),
 np.float64(2007.0),
 np.float64(2008.0),
 np.float64(2009.0),
 np.float64(2010.0),
 np.float64(2011.0),
 np.float64(2012.0),
 np.float64(2013.0),
 np.float64(2014.0),
 np.float64(2015.0),
 np.float64(2016.0),
 np.float64(2017.0),
 np.float64(2018.0),
 np.float64(2019.0),
 np.float64(2020.0),
 np.float64(2021.0),
 np.float64(2022.0),
 np.float64(2023.0),
 np.float64(2024.0),
 np.float64(2025.0)]

In [7]:
#create a new column with the season
def month_to_season(month):
    if pd.isna(month):
        return "unknown"
    month = int(month)
    if month in (12, 1, 2):
        return "winter"
    elif month in (3, 4, 5):
        return "spring"
    elif month in (6, 7, 8):
        return "summer"
    elif month in (9, 10, 11):
        return "autumn"
    else:
        return "unknown"
    
df["season"] = df["month"].apply(month_to_season)

In [8]:
print("the values are:", df["season"].unique())
print("the values are:", df["season"].value_counts())

the values are: ['spring' 'winter' 'summer' 'autumn']
the values are: season
spring    3307
autumn    2740
winter    2520
summer    1948
Name: count, dtype: int64


In [9]:
# Max day in each month
max_day_per_month = df.groupby('month')['day'].max()

print(max_day_per_month)

month
1.0     31.0
2.0     29.0
3.0     31.0
4.0     30.0
5.0     31.0
6.0     30.0
7.0     31.0
8.0     31.0
9.0     30.0
10.0    31.0
11.0    30.0
12.0    31.0
Name: day, dtype: float64


In [10]:
df.language.value_counts()

language
it    10154
en      261
es       47
pt       15
fr       10
ca       10
tl        4
so        3
de        2
ro        2
sw        1
sl        1
id        1
vi        1
et        1
sv        1
no        1
Name: count, dtype: int64

# Song category

## lyrics → lyric_topic

In [11]:
# Clean lyrics

In [12]:
import re
import warnings
from typing import Optional, Set, List, Dict

warnings.filterwarnings("ignore")

# 1. Text Normalization Config
CURLY_MAP = {
    "’": "'",
    "‘": "'",
    "“": '"',
    "”": '"',
    "—": "-",
    "–": "-",
    "…": "...",
}

SECTION_LINE_RE = re.compile(r"^\s*\[.*?\]\s*$", re.IGNORECASE)
BRACKET_SECTION_RE = re.compile(r"\[.*?\]")
HYPHEN_REPEAT_RE = re.compile(r"\b(\w{1,3})(?:-\1)+\b", re.IGNORECASE)

LYRICS_FILLER_WORDS = {
    "yeah", "yea", "yay",
    "whoo", "woo", "whoa", "woah",
    "oh", "ooh", "ah", "uh", "mm", "mmm", "hm", "hmm",
    "la", "na", "da",
    "hey", "yo",
}

ITALIAN_CONTRACTIONS = {
    "l'": "lo ",
    "un'": "una ",
    "dell'": "della ",
    "all'": "alla ",
    "nell'": "nella ",
    "sull'": "sulla ",
    "com'è": "come è",
    "dov'è": "dove è",
    "dov’": "dove ",
}

EXTRA_STOPWORDS = {

    # Generic lyric junk
    "lyric", "lyrics", "song", "instrumental",

    # Noise
    "su", "da", "con", "il", "lo", "la", "le", "gli",
    "non", "che", "come", "una", "sono",
}

# 2. Basic Text Normalization
def normalize_unicode(text: str) -> str:
    for bad, good in CURLY_MAP.items():
        text = text.replace(bad, good)
    return text

def normalize_repetitions(text: str) -> str:
    return HYPHEN_REPEAT_RE.sub(r"\1", text)

def remove_parenthetical_adlibs(text: str, max_len: int = 15) -> str:
    return re.sub(rf"\([^)]{{0,{max_len}}}\)", " ", text)

def expand_italian_contractions(text: str) -> str:
    for c, f in ITALIAN_CONTRACTIONS.items():
        text = text.replace(c, f)
    return text

def expand_contractions(text: str, language: str) -> str:
    if language == "en":
        try:
            import contractions
            return contractions.fix(text)
        except Exception:
            return text
    if language == "it":
        return expand_italian_contractions(text)
    return text

def remove_section_tags(text: str) -> str:
    return "\n".join(
        line for line in text.splitlines()
        if not SECTION_LINE_RE.match(line)
    )

def basic_pre_normalization(text: str, language: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.strip().strip('"').strip("'")
    if not text:
        return ""

    text = normalize_unicode(text)
    text = expand_contractions(text, language)
    text = text.lower()
    text = normalize_repetitions(text)
    text = remove_parenthetical_adlibs(text)
    text = BRACKET_SECTION_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

# 3. spaCy Utilities
def safe_load_spacy(model: str):
    try:
        import spacy
        return spacy.load(model, disable=("ner", "textcat"))
    except Exception:
        return None

def build_stopwords(nlp) -> Set[str]:
    if nlp is None:
        return set()
    return set(nlp.Defaults.stop_words)

# 4. Token-level helpers
def compress_token_runs(tokens: List[str], max_run: int = 1) -> List[str]:
    out, prev, count = [], None, 0
    for t in tokens:
        if t == prev:
            count += 1
        else:
            prev, count = t, 1
        if count <= max_run:
            out.append(t)
    return out

# 5. Multilingual Cleaning Function (FINAL)
def clean_text_multilingual(
    text: str,
    language: str,
    nlp_pipelines: Dict[str, object],
    stopwords_map: Dict[str, Set[str]],
) -> str:

    if not isinstance(text, str):
        return ""

    language = str(language).lower().strip()

    # ---------- BASIC CLEANING ----------
    text = basic_pre_normalization(text, language)
    text = remove_section_tags(text)

    if not text:
        return ""

    # ---------- NLP PIPELINE ----------
    nlp = nlp_pipelines.get(language)
    stopwords = stopwords_map.get(language, set())

    if nlp is None:
        # safe fallback for other languages
        return re.sub(r"\s+", " ", text).strip()

    try:
        doc = nlp(text)
    except Exception:
        return text

    tokens: List[str] = []

    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue

        form = tok.lemma_.lower().strip()

        if not form or len(form) < 3:
            continue

        if form.isdigit():
            continue
        
        if form in stopwords:
            continue

        if form in LYRICS_FILLER_WORDS:
            continue

        tokens.append(form)

    tokens = compress_token_runs(tokens)
    return " ".join(tokens)


In [13]:
nlp_en = safe_load_spacy("en_core_web_sm")
nlp_it = safe_load_spacy("it_core_news_sm")

nlp_pipelines = {
    "en": nlp_en,
    "it": nlp_it,
}

stopwords_map = {
    "en": build_stopwords(nlp_pipelines["en"]) | EXTRA_STOPWORDS,
    "it": build_stopwords(nlp_pipelines["it"]) | EXTRA_STOPWORDS,
}

In [14]:
df["lyrics_clean"] = df.apply(
    lambda r: clean_text_multilingual(
        r["lyrics"],
        r["language"],
        nlp_pipelines,
        stopwords_map
    ),
    axis=1
)

In [15]:
#prepare document
docs = df["lyrics_clean"].dropna().tolist()

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

tfidf = TfidfVectorizer(
    min_df=10,             
    max_df=0.45,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Zàèéìòù']{3,}\b",
    sublinear_tf=True,
    norm="l2"
)

X_tfidf = tfidf.fit_transform(df["lyrics_clean"].dropna())
features = tfidf.get_feature_names_out()


N_TOPICS = 8

nmf = NMF(
    n_components=N_TOPICS,
    random_state=42,
    init="nndsvd",
    max_iter=500
)

W = nmf.fit_transform(X_tfidf)
H = nmf.components_

In [17]:
def show_topics(H, feature_names, top_n=15):
    topics = {}
    for i, topic_vec in enumerate(H):
        top_idx = np.argsort(topic_vec)[::-1][:top_n]
        topics[i] = [feature_names[j] for j in top_idx]
    return topics

topics = show_topics(H, features)

for k, v in topics.items():
    print(f"\nTopic {k}: {v}")


Topic 0: ['pensare', 'sai', 'dovere', 'sentire', 'credere', 'chiedere', 'ciò', 'cambiare', 'dare', 'perdere', 'capire', 'parlare', 'fare', 'cercare', 'guardare']

Topic 1: ['soldo', 'ehi', 'gang', 'baby', 'fumare', 'giro', 'fumo', 'bitch', 'luke', 'sick', 'sick luke', 'chiamare', 'parlare', 'okay', 'yah']

Topic 2: ['nun', 'cchiù', 'pecché', 'int', 'llo', 'sulo', 'aggio', 'comme', 'maje', 'nno', 'chello', 'stu', 'tenere', 'nuje', 'sempere']

Topic 3: ['you', 'the', 'like', 'know', 'and', 'love', 'fuck', 'want', 'that', 'this', 'time', 'your', 'with', 'for', 'bitch']

Topic 4: ['lyricscome soon', 'lyricscome', 'soon', 'dame', 'dream', 'dope', 'world', 'chance', 'love', 'money', 'feed', 'need', 'ego', 'coming', 'stock']

Topic 5: ['rap', 'cazzo', 'con', 'tipo', 'flow', 'merda', 'disco', 'rapper', 'gente', 'culo', 'roba', 'rima', 'pezzo', 'beat', 'mano']

Topic 6: ['notte', 'cielo', 'occhio', 'cuore', 'sole', 'luce', 'sogno', 'mare', 'amore', 'stella', 'buio', 'strada', 'vento', 'luna', 

In [18]:
df_category = df.loc[df["lyrics_clean"].notna()].copy()

df_category["lyrics_topic_id"] = W.argmax(axis=1)
df_category["lyrics_topic_conf"] = W.max(axis=1)

In [19]:
TOPIC_LABELS = {
    0: "reflection_life",
    1: "trap_street",
    2: "dialect_rap",
    3: "english_pop_trap",
    4: "aspiration_generic",
    5: "rap_identity",
    6: "love_poetic",
    7: "spanish",
}

df_category["lyrics_topics"] = df_category["lyrics_topic_id"].map(TOPIC_LABELS)

In [20]:
df_category["lyrics_topics"].value_counts()

lyrics_topics
rap_identity          3054
reflection_life       2575
love_poetic           2261
trap_street           1327
english_pop_trap       668
dialect_rap            412
spanish                192
aspiration_generic      26
Name: count, dtype: int64

## swear → lyric_tone

In [21]:
df_category["swear_ratio"] = (
    df_category["swear_IT"].fillna(0) +
    df_category["swear_EN"].fillna(0)
) / df_category["n_tokens"].clip(lower=1)

In [22]:
def lyric_tone(row):
    if row["explicit"] and row["swear_ratio"] > 0.02:
        return "aggressive"
    if row["swear_ratio"] > 0.01:
        return "street"
    return "clean"

df_category["lyric_tone"] = df_category.apply(lyric_tone, axis=1)

## audio signal → melody_label

In [23]:
AUDIO_COLS = [
    "bpm", "pitch", "loudness",
    "spectral_complexity", "flux",
    "flatness", "duration_ms"
]

X_audio = df_category[AUDIO_COLS].fillna(df_category[AUDIO_COLS].median())

In [24]:
from sklearn.preprocessing import StandardScaler

X_audio_scaled = StandardScaler().fit_transform(X_audio)

In [25]:
from sklearn.cluster import KMeans

N_AUDIO_CLUSTERS = 4

kmeans = KMeans(
    n_clusters=N_AUDIO_CLUSTERS,
    random_state=42,
    n_init=20
)

df_category["melody_cluster"] = kmeans.fit_predict(X_audio_scaled)

In [26]:
df_category["melody_cluster"].value_counts()

melody_cluster
0    4122
1    3356
3    2977
2      60
Name: count, dtype: int64

In [27]:
AUDIO_FEATURES = [
    "bpm",
    "pitch",
    "loudness",
    "spectral_complexity",
    "flux",
    "flatness",
    "duration_ms"
]

cluster_profile = (
    df_category
    .groupby("melody_cluster")[AUDIO_FEATURES]
    .mean()
    .round(2)
)

cluster_profile

,bpm,pitch,loudness,spectral_complexity,flux,flatness,duration_ms
melody_cluster,,,,,,,
0,114.14,2044.05,27.95,34.69,1.23,0.86,212558.67
1,114.90,2606.86,28.80,25.81,1.35,0.87,190142.49
2,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00,206429.55
3,114.89,2138.44,15.08,19.80,1.20,0.85,201556.08


In [28]:
MELODY_LABELS = {
    0: "chill",
    1: "hard",
    2: "energetic",
    3: "unknown"
}

df_category["melody_label"] = df_category["melody_cluster"].map(MELODY_LABELS)

In [29]:
df_category["melody_label"].value_counts()

melody_label
chill        4122
hard         3356
unknown      2977
energetic      60
Name: count, dtype: int64

## combine → song_category

In [30]:
df_category["song_category"] = (
    "<" + df_category["lyrics_topics"].astype(str)
    + ">_<" + df_category["melody_label"]
    + ">_<" + df_category["lyric_tone"] + ">"
)

In [31]:
OUTPUT_COLS = [
    "title",
    "lyrics_topics",
    "lyric_tone",
    "melody_label",
    "song_category"
]

df_category[OUTPUT_COLS].head(20)

,title,lyrics_topics,lyric_tone,melody_label,song_category
0,​polka 2 :-/,rap_identity,aggressive,unknown,<rap_identity>_<unknown>_<aggressive>
1,POLKA,trap_street,aggressive,chill,<trap_street>_<chill>_<aggressive>
2,​britney ;-),trap_street,aggressive,hard,<trap_street>_<hard>_<aggressive>
3,CEO,trap_street,aggressive,hard,<trap_street>_<hard>_<aggressive>
4,LONDRA,love_poetic,clean,chill,<love_poetic>_<chill>_<clean>
5,BOHEME,love_poetic,clean,unknown,<love_poetic>_<unknown>_<clean>
6,LOBBY WAY,trap_street,street,chill,<trap_street>_<chill>_<street>
7,SLATT,trap_street,aggressive,chill,<trap_street>_<chill>_<aggressive>
8,MADE IN ITALY,love_poetic,clean,chill,<love_poetic>_<chill>_<clean>
9,ROSE & ROVI,trap_street,aggressive,chill,<trap_street>_<chill>_<aggressive>


In [32]:
df_category["song_category"].value_counts()

song_category
<love_poetic>_<chill>_<clean>             970
<reflection_life>_<chill>_<clean>         961
<reflection_life>_<unknown>_<clean>       794
<rap_identity>_<chill>_<clean>            749
<love_poetic>_<unknown>_<clean>           660
                                         ... 
<spanish>_<unknown>_<street>                2
<trap_street>_<energetic>_<aggressive>      1
<love_poetic>_<energetic>_<street>          1
<reflection_life>_<energetic>_<street>      1
<english_pop_trap>_<energetic>_<clean>      1
Name: count, Length: 76, dtype: int64

In [33]:
df_category.to_csv('/Users/huynhphuongchi/Desktop/Unipi/DSS/Module 2/LDS Data 2025-2026/tracks_cleaned_with_song_category.csv', index=False)